In [1]:
import sys
import os 
print("ori_workdir:", os.getcwd())
os.chdir(os.path.abspath('..'))
print("cur_workdir:", os.getcwd())
sys.path.append(os.getcwd())

ori_workdir: /root/autodl-tmp/rag-service/client
cur_workdir: /root/autodl-tmp/rag-service


In [2]:
import requests
import json
from config import cfg

Env is prod


In [4]:
# # 服务器地址
BASE_URL = f"http://{cfg.server.host}:{cfg.server.port}"
print(BASE_URL)

http://0.0.0.0:50051


### 健康检查

In [10]:
# 1. 健康检查示例
def check_health():
    url = f"{BASE_URL}/health"
    response = requests.get(url)
    return response.json()

health_result = check_health()
print("Health check:", health_result)

Health check: {'status': 'healthy'}


### 插入文本

In [5]:
# 2. 插入文本示例
def insert_text(text):
    url = f"{BASE_URL}/insert"
    headers = {"Content-Type": "application/json"}
    data = {
        "text": text
    }
    response = requests.post(url, json=data, headers=headers)
    return response.json()

text = "我的名字叫无师大模型"
insert_result = insert_text(text)
print("Insert result:", insert_result)

Insert result: {'status': 'success', 'data': None, 'message': 'Text inserted successfully'}


### 插入文件

In [6]:
# 3. 插入文件示例
def insert_file(file_path):
    url = f"{BASE_URL}/insert_file"
    with open(file_path, 'rb') as file:
        files = {'file': file}
        response = requests.post(url, files=files)
    return response.json()

file_path = 'resources/raw_docs/xiaolu.txt'
file_result = insert_file(file_path)
print("File insert result:", file_result)

File insert result: {'status': 'success', 'data': None, 'message': '成功处理文件 xiaolu.txt 并插入内容'}


In [7]:
# excel 文件
file_path = 'resources/raw_docs/XiaoMing-Life.xlsx'
file_result = insert_file(file_path)
print("File insert result:", file_result)

File insert result: {'status': 'success', 'data': None, 'message': '成功处理文件 XiaoMing-Life.xlsx 并插入内容'}


### 查询示例

In [5]:
# 4. 查询示例
def make_query(query_text, mode="hybrid", only_need_context:bool=False):
    url = f"{BASE_URL}/query/full"
    headers = {"Content-Type": "application/json"}
    data = {
        "query": query_text,
        "mode": mode,
        "only_need_context": only_need_context
    }
    response = requests.post(url, json=data, headers=headers)
    return json.dumps(response.json(), indent=4, ensure_ascii=False)

# query = "你是谁"
# query = "小陆是谁"
# query = "小明是谁"
# query = "小明和小陆有什么关系"
# query = "小明的父亲是谁"
query = "企业名称是什么"
mode = ["naive","local","global","hybrid"][0] 
only_need_context = False
query_result = make_query(query, mode, only_need_context)
print("Query result:", query_result)

print('-'*42)
print(json.loads(query_result)['data'])

Query result: {
    "status": "success",
    "data": "<think>\n好的，我现在要处理用户的问题：“企业名称是什么”。首先，我需要仔细查看提供的文档块，看看是否有提到企业名称。\n\n文档块里列出了属性名称和对应的属性值。第一个属性是“名称”，对应的属性值是“无师大模型”。看起来“无师大模型”可能是模型或产品的名称，而不是企业名称。不过，用户询问的是企业名称，而文档中并没有明确提到公司或企业的名字。可能用户误以为文档中包含企业信息，但实际上提供的文档主要描述的是一个名为“无师大模型”的模型架构和功能。\n\n接下来，我需要确认文档中的其他部分是否有任何关于企业的线索。检查所有属性和模块描述，发现都是技术细节，如架构类型、核心功能、知识库规模等，没有涉及公司名称或相关的企业信息。因此，根据提供的文档内容，无法确定企业名称。\n\n用户可能在提问时假设文档中有企业名称，但根据现有信息，必须如实回复无法提供。同时，要确保不添加文档外的假设或信息，严格遵守响应规则。因此，正确的回答应该是告知用户文档中未提供企业名称，并简要说明文档所涉及的内容重点。\n</think>\n\n根据提供的文档内容，并未明确提及具体的**企业名称**。文档主要描述了一个名为“无师大模型”的技术架构及其相关功能模块，例如认知引擎、记忆中枢、逻辑处理器等。该模型的特点包括万亿级跨领域知识库、检索增强生成（RAG）、对比学习与残差反馈优化等技术特性，但未涉及企业或公司的名称信息。\n\n若需了解企业名称，建议补充与具体公司或产品相关的文档内容。当前文档聚焦于技术细节，例如：\n- **架构类型**：检索增强生成（RAG）\n- **核心功能**：动态知识检索、多轮对话推理\n- **技术实现**：Transformer+图注意力网络、混合检索架构（Hybrid RAG）\n- **性能指标**：上下文窗口 16K tokens、检索延迟 <300ms\n\n请确认是否需要进一步解释该模型的技术特性。",
    "message": null
}
------------------------------------------
<think>
好的，我现在要处理用户的问题：“企业名称是什么”。首先，我需要仔细查看提供的文档块，看看是否有提到

### 流式查询

In [10]:
import requests
from IPython.display import display, clear_output

# 查询参数
query_data = {
    "query": "小明的父亲是谁",
    "mode": "hybrid",
    "only_need_context": False
}
url = BASE_URL + "/query/stream"

# 流式客户端
def stream_client():
    # 创建一个显示区域用于实时更新
    output_area = display("", display_id=True)
    
    try:
        # 发送POST请求，启用流式响应
        response = requests.post(
            url,
            json=query_data,
            stream=True  # 启用流式响应
        )
        
        # 检查响应状态
        if response.status_code != 200:
            print(f"Error: Received status code {response.status_code}")
            return
        
        # 逐块读取流式内容
        for chunk in response.iter_content(chunk_size=None, decode_unicode=True):
            if chunk:  # 过滤空块
                # 直接处理原始内容
                if chunk.startswith('ERROR:'):
                    print(f"\nError: {chunk[6:]}")
                    break
                else:
                    # 实时更新显示
                    # clear_output(wait=True)
                    output_area.update(f"Received: {chunk}")
                    print(chunk, end='', flush=True)
        
        response.close()
        # print("\nStream completed")
        
    except Exception as e:
        print(f"Client error: {str(e)}")

# 执行客户端
stream_client()

'Received: 。'

<think>
好，我需要回答用户的问题：“小明的父亲是谁”。首先，我要查看提供的知识库中是否有相关信息。知识库里有几个实体和关系，比如“小陆”、“银行”、“手工糕点店”等等。

在知识库的实体部分，只提到了“小陆”，没有提到任何其他人物如小明或他父亲的信息。此外，在关系部分也没有任何关于“小明”的信息，所以没有任何数据可以指向小明的父亲是谁。

根据知识库中的信息范围，我只能回复说这个问题不在当前的知识库范围内，或者具体信息缺失。因此，我需要礼貌地告知用户没有相关资料，并建议他们查看更广泛的信息来源。

最后，确保回应简洁明了，符合格式要求，不添加额外信息。
</think>

根据您提供的知识库内容，目前并没有提到“小明”或其父亲的相关信息。如果您能提供更多背景或详细信息，我可以帮助您进一步分析。